# Register Qwen3-VL as a Databricks Model Serving Endpoint

One-time (or occasional, if the model changes) infrastructure setup —
**not** part of the bundle-managed ingestion pipeline. Deploys
`Qwen/Qwen3-VL-4B-Instruct` (the Hugging Face Transformers release, *not*
the Ollama GGUF build used for local testing) as a custom, GPU-backed
Model Serving endpoint, so the RAG ingestion pipeline can call it the same
way it called every model in the OCR spike test — an OpenAI-style chat
completions API accepting image input.

**Prerequisites:**
- Attach this notebook to a cluster with a **GPU** — a classic all-purpose
  cluster with a single A10 node works fine (simpler, more transparent
  DBU-based pricing than serverless GPU compute); this is a one-time/
  occasional interactive setup task, so spin it up, run the notebook, and
  terminate it when done. This cluster is only for *running this
  notebook* — it is separate from the deployed serving endpoint's own GPU
  compute (step 4 below), which is Databricks-managed and billed
  independently (scale-to-zero).
- `mlflow>=3.12.0` and `databricks-sdk>=0.102.0` (installed below).

**Why plain `transformers`, not vLLM.** An earlier version of this
notebook used vLLM (Databricks' documented pattern for custom
vision-language model serving, and it gives an OpenAI-compatible API for
free). In practice it hit a chain of environment-specific issues — a FIPS
crash on classic clusters, a driver-too-old error from an unpinned
dependency pulling a too-new CUDA release, then missing CUDA development
headers needed to JIT-compile one of vLLM's optional accelerated
sampling kernels. That last class of problem kept recurring: this
environment ships pip-installable CUDA *runtime* libraries but not a full
CUDA *development* toolkit, which vLLM's optional kernels assume. vLLM's
actual value — continuous batching, PagedAttention, high-throughput
concurrent serving — matters for busy interactive chat traffic; this is a
low-concurrency batch OCR job (triggered weekly or manually, processing a
few dozen pages at a time), so none of that throughput optimization was
being used anyway. Plain `transformers.generate()` needs no JIT-compiled
kernels at all, at the cost of writing the OpenAI-compatible request/
response shaping ourselves (step 3) rather than getting it for free.

**A note on confidence for step 3.** Databricks Model Serving passes the
raw request body through to a custom pyfunc model's `predict()`, but the
*exact* wire shape (a bare dict vs. a list vs. a pandas-DataFrame-like
object) isn't something I could fully verify without a live test. The
`predict()` method below parses defensively and the notebook's own local
smoke test (step 2) and end-to-end validation (step 5) are structured to
catch a wrong assumption quickly rather than silently.


In [ ]:
# Exact pins, not minimums -- >=X.Y.Z resolves to whatever the latest
# compatible release is at install time, which can silently drift to a
# newer, differently-behaved version later (this bit us hard earlier in
# this notebook's history: an unpinned vllm>=0.11.0 resolved to a release
# many versions newer that needed a CUDA version this environment didn't
# have). Versions below are the ones actually confirmed working in this
# environment during development -- accelerate and torchvision are the
# exceptions, newly added for this transformers-based rewrite and not
# yet confirmed (torchvision: needed by Qwen3-VL's AutoProcessor, missed
# in the first pass); pin both exactly once you see what versions this
# cell resolves them to.
%pip install "mlflow==3.15.1" "databricks-sdk==0.125.0" "huggingface_hub==0.34.4" "transformers==4.57.1" accelerate torchvision "hf_transfer==0.1.9" "pillow==12.3.0" "pdfplumber==0.11.10"
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "eliao")
dbutils.widgets.text("schema", "wnv_demo")
dbutils.widgets.text("model_name", "qwen3_vl_ocr")
dbutils.widgets.text("hf_model_id", "Qwen/Qwen3-VL-4B-Instruct")
dbutils.widgets.text("endpoint_name", "qwen3-vl-ocr")
# GPU_MEDIUM = 1x A10 (24GB) -- Databricks' own docs list this as the
# default tier for general inference, and their worked example for a
# *smaller* vision model (Qwen2.5-VL-3B) uses A10, not T4. Vision models
# need more headroom than a same-size text model: page-image inputs
# decode into a lot of vision tokens, which consume KV-cache memory on
# top of the vision encoder itself. GPU_SMALL (T4, 16GB) was the initial
# assumption in the design doc but wasn't actually validated -- corrected
# here.
dbutils.widgets.dropdown("workload_type", "GPU_MEDIUM", ["GPU_SMALL", "GPU_MEDIUM", "GPU_LARGE"])
# Only needed for the end-to-end validation cell at the bottom.
dbutils.widgets.text(
    "test_pdf_volume_path",
    "/Volumes/eliao/wnv_demo/documents/WNV-Outbreak-Communications-Toolkit-2025_508c.pdf",
)
dbutils.widgets.text("test_page", "5")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
model_name = dbutils.widgets.get("model_name").strip()
hf_model_id = dbutils.widgets.get("hf_model_id").strip()
endpoint_name = dbutils.widgets.get("endpoint_name").strip()
workload_type_str = dbutils.widgets.get("workload_type").strip()

uc_model_name = f"{catalog}.{schema}.{model_name}"
print(f"UC model:       {uc_model_name}")
print(f"Endpoint:       {endpoint_name}")
print(f"Source model:   {hf_model_id}")
print(f"GPU workload:   {workload_type_str}")

## 1. Download model weights

Downloads the actual Hugging Face Transformers release (safetensors) to
local (non-`/Workspace`) storage — large model weights don't belong on the
workspace filesystem, and this is a genuinely different artifact format
from the Ollama GGUF build already on your machine; that one is for
llama.cpp-style local inference, not compatible with the serving path here.


In [ ]:
import tempfile
from pathlib import Path

from huggingface_hub import snapshot_download

local_model_dir = Path(tempfile.mkdtemp()) / "model"
snapshot_download(repo_id=hf_model_id, local_dir=str(local_model_dir))
print(f"Downloaded {hf_model_id} to {local_model_dir}")

## 2. Smoke-test locally with `transformers`

Loads the model directly in-process and runs one generation — no
subprocess, no server, no ports. Confirms the model loads and generates
before wiring it into the pyfunc wrapper below.


In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

model = Qwen3VLForConditionalGeneration.from_pretrained(
    str(local_model_dir), dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained(str(local_model_dir))

# Plain text smoke test -- the full image-input path gets exercised by
# the end-to-end validation cell in step 5, against the real endpoint.
# Content must be a list of typed parts, even for text-only messages --
# Qwen3-VL's chat template doesn't accept a bare string here (confirmed
# by a real run: TypeError from iterating over the string's characters).
messages = [{"role": "user", "content": [{"type": "text", "text": "Reply with exactly: OK"}]}]
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

generated_ids = model.generate(**inputs, max_new_tokens=10)
trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
print(output_text)

# Free GPU memory before the model gets loaded again inside the logged
# pyfunc model's load_context() below.
del model, processor
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## 3. Log to MLflow

A custom `PythonModel` rather than MLflow's built-in chat-model helpers
(`ChatModel` is deprecated as of MLflow 3.0, and it's unclear whether its
replacement's structured message types support multimodal content parts)
— this parses the incoming OpenAI-style request and returns an
OpenAI-style response explicitly, matching exactly what the ingestion
pipeline and the OCR spike test already send/expect.


In [ ]:
import base64
import io
import json
import os

import mlflow
from PIL import Image

# Must log+register from serverless GPU compute -- otherwise the model
# gets packaged with CPU dependencies and the GPU serving endpoint fails
# to start. Per Databricks' own reference notebook for this serving path.
if not os.environ.get("DATABRICKS_ACCELERATOR"):
    raise RuntimeError(
        "This must be logged+registered from serverless GPU compute, or "
        "the endpoint will be packaged with the wrong dependencies."
    )

mlflow.set_registry_uri("databricks-uc")


class Qwen3VLChatModel(mlflow.pyfunc.PythonModel):
    """Custom pyfunc wrapping Qwen3-VL for OpenAI-compatible chat serving."""

    def load_context(self, context):
        from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

        model_path = context.artifacts["model_dir"]
        self.model = Qwen3VLForConditionalGeneration.from_pretrained(
            model_path, dtype="auto", device_map="auto"
        )
        self.processor = AutoProcessor.from_pretrained(model_path)

    def predict(self, context, model_input, params=None):
        # Defensive parsing -- see the confidence note in step 3's markdown
        # above for why this isn't a single trusted shape. Confirmed by a
        # real run: Model Serving passed the request as a raw JSON string
        # rather than a pre-parsed dict, at more than one level.
        request = self._maybe_parse_json(model_input)
        if isinstance(request, list):
            request = self._maybe_parse_json(request[0])
        if hasattr(request, "to_dict"):  # pandas Series/DataFrame row
            request = request.to_dict()

        messages = self._maybe_parse_json(request["messages"])
        max_new_tokens = request.get("max_tokens", 512)
        temperature = request.get("temperature", 0.0)

        chat_messages = self._to_chat_format(messages)
        inputs = self.processor.apply_chat_template(
            chat_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(self.model.device)

        generated_ids = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0.0,
            temperature=temperature if temperature > 0.0 else None,
        )
        trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = self.processor.batch_decode(
            trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        return {
            "choices": [
                {
                    "index": 0,
                    "message": {"role": "assistant", "content": output_text},
                }
            ]
        }

    @staticmethod
    def _maybe_parse_json(value):
        """Model Serving can hand a custom pyfunc model raw JSON strings
        instead of pre-parsed dicts/lists, depending on request wire shape.
        Idempotent on already-parsed input."""
        if isinstance(value, str):
            return json.loads(value)
        return value

    @staticmethod
    def _to_chat_format(messages: list[dict]) -> list[dict]:
        """OpenAI-style messages (content as a list of text/image_url parts)
        -> Qwen3-VL's chat-template content format
        ({"type": "image", "image": <PIL.Image>} / {"type": "text", ...}).
        """
        chat_messages = []
        for msg in messages:
            content = msg["content"]
            if isinstance(content, str):
                # Qwen3-VL's chat template requires a list of typed parts,
                # not a bare string, even for plain text -- confirmed by a
                # real run hitting this exact gap in the smoke test above.
                content = [{"type": "text", "text": content}]

            parts = []
            for part in content:
                if part["type"] == "text":
                    parts.append({"type": "text", "text": part["text"]})
                elif part["type"] == "image_url":
                    url = part["image_url"]["url"]
                    if url.startswith("data:"):
                        b64_data = url.split(",", 1)[1]
                        image = Image.open(io.BytesIO(base64.b64decode(b64_data)))
                    else:
                        image = url  # http(s) URL -- processor fetches it directly
                    parts.append({"type": "image", "image": image})
            chat_messages.append({"role": msg["role"], "content": parts})
        return chat_messages


# Unity Catalog Model Registry requires an explicit input/output
# signature on every model -- built from representative example dicts
# matching our actual request/response shape, not by invoking the
# model itself (which isn't loaded at this point; load_context() only
# runs when the model is actually served).
from mlflow.models import infer_signature

input_example = {
    "messages": [
        {"role": "user", "content": [{"type": "text", "text": "Hello"}]}
    ],
    "max_tokens": 512,
    "temperature": 0.0,
}
output_example = {
    "choices": [
        {"index": 0, "message": {"role": "assistant", "content": "Hi there!"}}
    ]
}
signature = infer_signature(input_example, output_example)

model_info = mlflow.pyfunc.log_model(
    name=model_name,
    python_model=Qwen3VLChatModel(),
    signature=signature,
    input_example=input_example,
    artifacts={"model_dir": str(local_model_dir)},
    metadata={"task": "llm/v1/chat"},
    # Matches the pins in the install cell above exactly, so the
    # deployed endpoint's container gets the same versions actually
    # tested locally, not independently-resolved ones.
    extra_pip_requirements=[
        "mlflow==3.15.1",
        "transformers==4.57.1",
        "accelerate",  # pin exactly here too once confirmed above
        "torchvision",  # same -- pin exactly once confirmed above
        "pillow==12.3.0",
    ],
)
print(f"Logged: {model_info.model_uri}")

In [ ]:
# env_pack is required -- Databricks' custom LLM/chat serving depends on
# Serverless Optimized Deployments; without it the endpoint does not work,
# per Databricks' own reference notebook for this exact serving path.
model_version = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=uc_model_name,
    env_pack="databricks_model_serving",
)
print(f"Registered: {uc_model_name} version {model_version.version}")

## 4. Create the GPU serving endpoint

`scale_to_zero_enabled=True` — matches the design decision that this
endpoint should cost nothing between ingestion runs (the ingestion job is
scheduled/manually-triggered, not continuous). First call after being idle
will have a cold-start delay while the endpoint reloads the model.


In [ ]:
from datetime import timedelta

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)

w = WorkspaceClient()

config = EndpointCoreConfigInput(
    served_entities=[
        ServedEntityInput(
            entity_name=uc_model_name,
            entity_version=str(model_version.version),
            workload_type=getattr(ServingModelWorkloadType, workload_type_str),
            workload_size="Small",
            scale_to_zero_enabled=True,
        )
    ]
)

print(f"Creating endpoint '{endpoint_name}' -- can take 10+ minutes for a first deploy.")
w.serving_endpoints.create_and_wait(name=endpoint_name, config=config, timeout=timedelta(minutes=45))
print("Endpoint ready.")

## 5. Validate end-to-end

Same request shape validated in the OCR spike test (`notebooks/rag_ocr_spike.ipynb`)
— confirms this endpoint is a drop-in for `WNV_VISION_ENDPOINT` with no
other code changes needed.


In [ ]:
import base64
import io
import json
from urllib import request as urlrequest

import pdfplumber

test_pdf_path = dbutils.widgets.get("test_pdf_volume_path").strip()
test_page_num = int(dbutils.widgets.get("test_page").strip())

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
databricks_host = context.apiUrl().get().removeprefix("https://").removesuffix("/")
databricks_token = context.apiToken().get()

with pdfplumber.open(test_pdf_path) as pdf:
    page = pdf.pages[test_page_num - 1]
    image = page.to_image(resolution=300)
    buf = io.BytesIO()
    image.original.save(buf, format="PNG")
    image_bytes = buf.getvalue()

b64 = base64.b64encode(image_bytes).decode()
payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Extract all text from this document page as clean markdown."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ],
        }
    ],
    "max_tokens": 2000,
    "temperature": 0.0,
}
url = f"https://{databricks_host}/serving-endpoints/{endpoint_name}/invocations"
req = urlrequest.Request(
    url,
    data=json.dumps(payload).encode(),
    headers={
        "Authorization": f"Bearer {databricks_token}",
        "Content-Type": "application/json",
    },
    method="POST",
)
with urlrequest.urlopen(req, timeout=120) as resp:
    result = json.loads(resp.read().decode())

print(result["choices"][0]["message"]["content"])

## Done

Set `WNV_VISION_ENDPOINT` to the value of `endpoint_name` above (default
`qwen3-vl-ocr`) wherever the ingestion pipeline configures it. The endpoint
is scale-to-zero, so the first call after being idle will be slow (model
reload) — expected, not a bug.
